<a href="https://colab.research.google.com/github/anandhij123/Python/blob/main/agents/Orchestration_langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain_core langchain_openai langchain_community langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain_core
    Found existing installation: langchain-core 1.2.16
    Uninstalling langchain-core-1.2.16:
      Successfully uninstalled langchain-core-1.2.16
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google

In [2]:
import os
import operator
from typing import TypedDict, Annotated, List
from dotenv import load_dotenv

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

In [3]:
llm = ChatOpenAI(openai_api_base = "https://openrouter.ai/api/v1", openai_api_key = "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", model = "stepfun/step-3.5-flash:free",temperature=0.7) #max_tokens=100

In [4]:
# --- Agent State Definition ---
# This TypedDict defines the structure of our application's state.
# It's how data is passed between the nodes in our graph.
class AgentState(TypedDict):
    task: str
    plan: str
    code: str
    execution_result: str
    test_result: str
    report: str
    # This special key 'messages' is for managing conversational history if needed.
    messages: Annotated[List[str], operator.add]

--- Agent Nodes ---
Each agent is represented by a function (a "node" in the graph).
These functions take the current state, perform their action, and return a dictionary
with the updated state values.

In [5]:
def manager_agent(state: AgentState) -> dict:
    """
    Manager agent: Receives the initial task and creates a high-level plan.
    """
    print("---MANAGER---")
    prompt = ChatPromptTemplate.from_template(
        """
        You are a project manager. Your role is to understand a user's programming request
        and create a clear, simple plan for your team.

        User Request: {task}

        Based on this, create a two-step plan:
        1.  A clear instruction for the Developer agent to write the Python code.
        2.  A clear instruction for the Tester agent to test the written code.
        """
    )
    chain = prompt | llm
    result = chain.invoke({"task": state["task"]})
    print(f"Manager's Plan:\n{result.content}")
    return {"plan": result.content}


In [6]:

def developer_agent(state: AgentState) -> dict:
    """
    Developer agent: Writes Python code based on the manager's plan and ensures it's executable.
    """
    print("---DEVELOPER---")

    # Use a loop to allow the agent to self-correct on syntax errors
    attempts = 0
    max_attempts = 3
    while attempts < max_attempts:
        print(f"Attempt {attempts + 1} of {max_attempts}")
        prompt = ChatPromptTemplate.from_template(
            """
            You are a Python developer. Your task is to write clean, executable Python code.
            Only output the raw Python code, without any markdown formatting or explanations.

            Manager's Plan:
            {plan}

            Current Code (if any, for refinement):
            <code>
            {code}
            </code>

            Execution Error (if any):
            {error}

            Write the Python code for the user's request: {task}
            """
        )
        chain = prompt | llm
        result = chain.invoke({
            "plan": state["plan"],
            "task": state["task"],
            "code": state.get("code", ""),
            "error": state.get("execution_result", "")
        })

        code = result.content.strip().replace("```python", "").replace("```", "")
        print(f"Developer's Code:\n{code}")

        # Execute the code to check for syntax errors
        try:
            exec(code)
            print("Code executed successfully (no syntax errors).")
            return {"code": code, "execution_result": "Code has no syntax errors."}
        except Exception as e:
            print(f"Execution failed with error: {e}")
            # If execution fails, update state and retry
            state["code"] = code
            state["execution_result"] = str(e)
            attempts += 1

    print("Developer failed to produce valid code after multiple attempts.")
    return {"code": code, "execution_result": "Failed to write executable code."}

In [7]:
def tester_agent(state: AgentState) -> dict:
    """
    Tester agent: Executes the code and reports on its functionality.
    """
    print("---TESTER---")
    prompt = ChatPromptTemplate.from_template(
        """
        You are a Quality Assurance (QA) tester. Your job is to execute the given Python code
        and report whether it works as expected based on the original task.

        Original Task: {task}
        Manager's Plan: {plan}

        Python Code to Test:
        <code>
        {code}
        </code>

        Provide a brief report on the outcome. Did the code run? Did it produce the expected result?
        """
    )
    chain = prompt | llm
    print("Executing code for testing...")
    try:

        exec(state["code"])
        test_outcome = "Code executed successfully. The functionality appears to work as per the task."
        print(test_outcome)
    except Exception as e:
        test_outcome = f"Code execution failed during testing. Error: {e}"
        print(test_outcome)

    return {"test_result": test_outcome}

In [9]:
def reporter_agent(state: AgentState) -> dict:
    """
    Reporter agent: Compiles the final report summarizing the entire process.
    """
    print("---REPORTER---")
    prompt = ChatPromptTemplate.from_template(
        """
        You are a reporting agent. Your task is to create a final, comprehensive summary
        of the entire development process. Combine all the information provided into a
        clear and concise report.

        **Initial Task:**
        {task}

        **Manager's Plan:**
        {plan}

        **Final Python Code:**
        <code>
        {code}
        </code>

        **Developer's Log:**
        {execution_result}

        **Tester's Report:**
        {test_result}

        Please generate the final summary report.
        """
    )
    chain = prompt | llm
    result = chain.invoke(state)
    print(f"---FINAL REPORT---\n{result.content}")
    return {"report": result.content}

In [10]:
# --- Graph Definition ---
# This is where we define the workflow (the directed graph).

# 1. Initialize the StateGraph
workflow = StateGraph(AgentState)

# 2. Add nodes to the graph. Each node is an agent function.
workflow.add_node("manager", manager_agent)
workflow.add_node("developer", developer_agent)
workflow.add_node("tester", tester_agent)
workflow.add_node("reporter", reporter_agent)

# 3. Define the edges that connect the nodes, determining the flow.
workflow.add_edge("manager", "developer")
workflow.add_edge("developer", "tester")
workflow.add_edge("tester", "reporter")
workflow.add_edge("reporter", END) # The reporter is the final step.

# 4. Set the entry point for the graph.
workflow.set_entry_point("manager")

# 5. Compile the graph into a runnable application.
app = workflow.compile()



In [ ]:
# --- Main Execution Block ---
if __name__ == "__main__":
    initial_task = input("Please enter the programming task for the agent team: ")

    # The input to the graph is a dictionary with the initial state.
    inputs = {"task": initial_task, "messages": []}

    # Invoke the graph and stream the output.
    # The output will be the final state of the graph after all nodes have run.
    for output in app.stream(inputs):
        # The key is the name of the node that just ran.
        # The value is the state dictionary after that node ran.
        for key, value in output.items():
            print(f"Output from node '{key}':")
            print("---")
            print(value, sep="\n", end="\n---\n")

    # The final state can be accessed from the last event in the stream
    final_state = list(app.stream(inputs))[-1]
    final_report = final_state['reporter']['report']
    print("\n--- Summary ---")
    print(final_report)

Please enter the programming task for the agent team: develop a tic tac toe game website
---MANAGER---
Manager's Plan:
### **Project Plan: Tic Tac Toe Game Website**

---

#### **1. Developer Agent Instructions**
**Objective:** Build a functional two-player Tic Tac Toe game using Python (Flask) for the backend and HTML/CSS/JavaScript for the frontend.

**Key Requirements:**
- Implement a 3x3 game board.
- Support two players alternating turns (X and O).
- Detect win/draw conditions and display results.
- Reset the board after a game ends.
- Use Flask to serve the game page and handle move submissions via AJAX.

**Step-by-Step Tasks:**
1. **Set up Flask app:**
   - Create `app.py` with routes:
     - `GET /`: Serve the main HTML page.
     - `POST /move`: Accept a player's move (row, col), validate it, update board state, check for win/draw, and return JSON response with updated board and game status.
   - Use a server-side session or in-memory structure (e.g., a dictionary) to store th

INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)


Code executed successfully (no syntax errors).
Output from node 'developer':
---
{'code': '\nfrom flask import Flask, render_template_string, request, jsonify, session\n\napp = Flask(__name__)\napp.secret_key = \'your_secret_key_here\'  # Change this to a random secret key in production\n\ndef initialize_board():\n    return [[\'\', \'\', \'\'], [\'\', \'\', \'\'], [\'\', \'\', \'\']]\n\ndef check_win(board, player):\n    # Check rows\n    for row in board:\n        if all(cell == player for cell in row):\n            return True\n    # Check columns\n    for col in range(3):\n        if all(board[row][col] == player for row in range(3)):\n            return True\n    # Check diagonals\n    if all(board[i][i] == player for i in range(3)):\n        return True\n    if all(board[i][2-i] == player for i in range(3)):\n        return True\n    return False\n\ndef check_draw(board):\n    for row in board:\n        for cell in row:\n            if cell == \'\':\n                return False\

INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)
